In [5]:
from pathlib import Path
from pytubefix import YouTube
from pydub import AudioSegment
from pydub.utils import which

def download_youtube_audio(
    url: str,
    destination: str | Path,
    start_ms: int | None = None,
    end_ms: int | None = None,
    output_filename: str | None = None,
    bitrate: str = "192k",
) -> Path:
    # --- Ensure ffmpeg/ffprobe are available (adjust if needed) ---
    ffmpeg_path  = which("ffmpeg")  or r"C:\ffmpeg\bin\ffmpeg.exe"
    ffprobe_path = which("ffprobe") or r"C:\ffmpeg\bin\ffprobe.exe"
    AudioSegment.converter = ffmpeg_path
    AudioSegment.ffprobe   = ffprobe_path

    dest = Path(destination)
    dest.mkdir(parents=True, exist_ok=True)

    # 1) Download audio-only stream (container often .webm or .m4a)
    yt = YouTube(url)
    stream = yt.streams.filter(only_audio=True).first()
    downloaded_path = Path(stream.download(output_path=dest))

    # 2) Load without forcing a format; ffmpeg will detect it
    audio = AudioSegment.from_file(downloaded_path)

    # 3) Resolve trim bounds (ms)
    if start_ms is None: start_ms = 0
    if end_ms   is None: end_ms   = len(audio)

    # Clamp to valid range
    start_ms = max(0, start_ms)
    end_ms   = min(len(audio), end_ms)
    if start_ms >= end_ms:
        raise ValueError(f"Invalid trim window: start_ms={start_ms}, end_ms={end_ms}, length={len(audio)}")

    segment = audio[start_ms:end_ms]

    # 4) Output filename/path
    if not output_filename:
        # default name: original stem + trimmed seconds
        output_filename = f"{downloaded_path.stem}_{start_ms//1000}-{end_ms//1000}.mp3"
    out_path = dest / output_filename  # <-- file path, not directory

    # 5) Export to mp3
    segment.export(out_path, format="mp3", bitrate=bitrate)
    return out_path


In [6]:
out = download_youtube_audio(
    url="https://www.youtube.com/watch?v=BdX1XeSxeKQ&t=4863s",
    destination=r"D:\Block A 2\Audio",
    start_ms=20 * 60 * 1000,
    end_ms=40 * 60 * 1000,
    output_filename="متری شیش و نیم_20to40.mp3"
)
print("Saved to:", out)


Saved to: D:\Block A 2\Audio\متری شیش و نیم_20to40.mp3


In [8]:
out = download_youtube_audio(
    url="https://www.youtube.com/watch?v=8yX8KFa2NFo&t=1875s",
    destination=r"D:\Block A 2\Audio",
    start_ms=27 * 60 * 1000,
    end_ms=47 * 60 * 1000,
    output_filename="همرفیق_20to40.mp3"
)
print("Saved to:", out)

Saved to: D:\Block A 2\Audio\همرفیق_20to40.mp3


In [18]:
out = download_youtube_audio(
    url="https://www.youtube.com/watch?v=F1QvKIsd95I",
    destination=r"D:\Block A 2\Audio",
    start_ms=27 * 60 * 1000,
    end_ms=67 * 60 * 1000,
    output_filename="zan.mp3"
)
print("Saved to:", out)

Saved to: D:\Block A 2\Audio\zan.mp3


In [19]:
import re
import assemblyai as aai
import pandas as pd
from pathlib import Path

def assemblyai_to_excel(
    path: str | Path,
    api_key: str,
    xlsx_path: str | Path | None = None,
    *,
    keep_speaker:  True
) -> str:
    """
    Transcribe Persian (fa) audio with AssemblyAI Universal-2
    and save an Excel file where each sentence is one row.

    Parameters
    ----------
    path : audio/video file path
    api_key : your AssemblyAI API key
    xlsx_path : optional output .xlsx path; defaults to <audio>.xlsx
    keep_speaker : include speaker label column if available

    Returns
    -------
    str : path to the written .xlsx file
    """

    # 1) Transcribe with Universal-2, Persian forced
    aai.settings.api_key = api_key
    audio_path = Path(path)

    config = aai.TranscriptionConfig(
        # model="universal-2",      # <-- explicitly select Universal-2
        language_code="fa",       # Persian
        punctuate=True,
        format_text=True,
        speaker_labels=True       # gives utterances with speaker tags
        # You can add: word_boost=['...'], boost_param='high' if needed
    )

    transcript = aai.Transcriber().transcribe(str(audio_path), config=config)
    if transcript.error:
        raise RuntimeError(f"AssemblyAI error: {transcript.error}")

    # 2) Build rows: sentence-per-line (Persian-aware split)
    # sentence boundary: ., !, ?, Persian ؟, ellipsis …
    SENT_SPLIT = re.compile(r'(?<=[\.!\?؟…])\s+')

    rows = []

    def add_sentences(text: str, speaker: str | None):
        if not text:
            return
        for s in (seg.strip() for seg in SENT_SPLIT.split(text) if seg.strip()):
            if keep_speaker:
                rows.append({"line": len(rows) + 1, "speaker": speaker or "", "text": s})
            else:
                rows.append({"line": len(rows) + 1, "text": s})

    # Prefer utterances (with speaker labels); fallback to full text
    if getattr(transcript, "utterances", None):
        for u in transcript.utterances:
            add_sentences((u.text or "").strip(), getattr(u, "speaker", None))
    else:
        add_sentences((transcript.text or "").strip(), speaker=None)

    # 3) Save to Excel (.xlsx). Excel handles UTF-8 in xlsx natively.
    df = pd.DataFrame(rows, columns=(["line", "speaker", "text"] if keep_speaker else ["line", "text"]))
    out_xlsx = Path(xlsx_path) if xlsx_path else audio_path.with_suffix(".xlsx")
    out_xlsx.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(out_xlsx, index=False)  # uses openpyxl by default if installed
    return str(out_xlsx)


In [14]:
out_file = assemblyai_to_excel(
    r"D:\Block A 2\Audio\متری شیش و نیم_20to40.mp3",
    api_key="536ece37a704451c9b4e0b1872eadf70",
    xlsx_path=r"D:\Block A 2\CSV\متری شیش و نیم_20to40.xlsx",
    keep_speaker=True
)
print("Saved to:", out_file)


Saved to: D:\Block A 2\CSV\متری شیش و نیم_20to40.xlsx


In [17]:
out_file = assemblyai_to_excel(
    r"D:\Block A 2\Audio\همرفیق_20to40.mp3",
    api_key="536ece37a704451c9b4e0b1872eadf70",
    xlsx_path=r"D:\Block A 2\CSV\همرفیق_20to40.xlsx" ,
    keep_speaker=True
)
print("Saved to:", out_file)

Saved to: D:\Block A 2\CSV\همرفیق_20to40.xlsx


In [20]:
out_file = assemblyai_to_excel(
    r"D:\Block A 2\Audio\zan.mp3",
    api_key="536ece37a704451c9b4e0b1872eadf70",
    xlsx_path=r"D:\Block A 2\CSV\zan.xlsx" ,
    keep_speaker=True
)
print("Saved to:", out_file)

Saved to: D:\Block A 2\CSV\zan.xlsx


In [36]:
from pathlib import Path
import re, math, pandas as pd, whisper

# --- Persian-aware sentence splitter ---
_SENT_SPLIT = re.compile(r'(?<=[\.!\?؟…])\s+')

def _split_sentences(text: str):
    text = re.sub(r'\s+', ' ', (text or '')).strip()
    parts = [s.strip() for s in _SENT_SPLIT.split(text) if s.strip()]
    return parts or ([text] if text else [])

def _fmt(sec: float):
    if sec is None: return ""
    ms = int(round((sec - int(sec)) * 1000))
    s = int(sec) % 60; m = (int(sec)//60)%60; h = int(sec)//3600
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

def whisper_fa_to_excel(
    media_path: str | Path,
    out_xlsx: str | Path | None = None,
    *,
    model_name: str = "large-v3",
    beam_size: int = 5,
    fp16: bool = False,
    initial_prompt: str | None =
        "زبان: فارسی. نگارش رسمی؛ نشانه‌گذاری فارسی (، ؛ ؟ …)؛ "
        "از تکرار بی‌دلیل واژه‌ها خودداری کن."
) -> str:
    """
    Transcribe Persian audio to Excel with one sentence per row.
    Anti-repetition decoding settings included.
    """
    media_path = Path(media_path)
    if not media_path.exists():
        raise FileNotFoundError(media_path)

    # Resolve output path
    if out_xlsx is None:
        out_xlsx = media_path.with_suffix(".xlsx")
    else:
        out_xlsx = Path(out_xlsx)
        if out_xlsx.exists() and out_xlsx.is_dir():
            out_xlsx = out_xlsx / f"{media_path.stem}.xlsx"
    out_xlsx.parent.mkdir(parents=True, exist_ok=True)

    model = whisper.load_model(model_name)

    # --- First pass (deterministic beam search) ---
    result = model.transcribe(
        str(media_path),
        language="fa",
        beam_size=beam_size,
        temperature=0.0,
        best_of=1,
        condition_on_previous_text=False,  # key against loops
        initial_prompt=initial_prompt,
        fp16=fp16,
        no_speech_threshold=0.6,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0
    )

    # --- Fallback with temp sweep if needed ---
    crit = result.get("compression_ratio", 0) and result["compression_ratio"] > 2.4
    if (not result.get("text")) or crit:
        result = model.transcribe(
            str(media_path),
            language="fa",
            beam_size=beam_size,
            temperature=(0.0, 0.2, 0.4),
            condition_on_previous_text=False,
            initial_prompt=initial_prompt,
            fp16=fp16,
            no_speech_threshold=0.6,
            compression_ratio_threshold=2.4,
            logprob_threshold=-1.0
        )

    # --- Build rows ---
    rows = []
    line = 1
    for seg in result.get("segments", []) or []:
        sents = _split_sentences(seg.get("text", ""))
        n = max(1, len(sents))
        start, end = float(seg["start"]), float(seg["end"])
        dur = max(0.0, end - start); step = dur / n if n else 0.0
        for i, s in enumerate(sents):
            rows.append({
                "line": line,
                "start": _fmt(start + i*step),
                "end":   _fmt(start + (i+1)*step),
                "text_fa": s,
                "text_en": "",   # ready for translations
                "topic": "",     # extra annotation column
                "label": "",     # extra annotation column
                "notes": ""      # extra annotation column
            })
            line += 1

    df = pd.DataFrame(rows, columns=["line","start","end","text_fa","text_en","topic","label","notes"])
    df.to_excel(out_xlsx, index=False)
    return str(out_xlsx)


In [37]:
out = whisper_fa_to_excel(
    r"D:\Block A 2\Audio\متری شیش و نیم_20to40.mp3",
    out_xlsx=r"D:\Block A 2\Whisper\metri_6.csv",
    model_name="large-v3",   # best for Persian
    beam_size=5,
    fp16=False               # set True if you have CUDA GPU
)
print("Saved:", out)


out = whisper_fa_to_excel(
    r"D:\Block A 2\Audio\همرفیق_20to40.mp3",
    out_xlsx=r"D:\Block A 2\Whisper\refigh.csv",
    model_name="large-v3",   # best for Persian
    beam_size=5,
    fp16=False               # set True if you have CUDA GPU
)
print("Saved:", out)


out = whisper_fa_to_excel(
    r"D:\Block A 2\Audio\zan.mp3",
    out_xlsx=r"D:\Block A 2\Whisper\zan.csv",
    model_name="large-v3",   # best for Persian
    beam_size=5,
    fp16=False               # set True if you have CUDA GPU
)
print("Saved:", out)




Saved: D:\Block A 2\Whisper\metri_6.csv
Saved: D:\Block A 2\Whisper\refigh.csv


KeyboardInterrupt: 

In [30]:
from pathlib import Path
from typing import Optional, List, Union
import pandas as pd

def translate_fa_to_en_excel_range(
    in_path: Union[str, Path],
    out_path: Optional[Union[str, Path]] = None,
    *,
    fa_col: str,                     # required: Persian text column
    en_col: str = "en",
    start_line: int = 1,
    end_line: Optional[int] = None,
    extra_columns: Optional[List[str]] = None,
    model_name: str = "facebook/nllb-200-distilled-600M",  # ← use NLLB
    batch_size: int = 32,
) -> str:
    """Translate rows [start_line..end_line] from `fa_col` to `en_col` and save as Excel."""
    in_path = Path(in_path)
    if out_path is not None:
        out_path = Path(out_path)
    if not in_path.exists():
        raise FileNotFoundError(in_path)
    if not fa_col.strip():
        raise ValueError("fa_col must be a non-empty column name.")

    # --- deps ---
    try:
        import torch
        from transformers import pipeline, AutoTokenizer
    except Exception as e:
        raise RuntimeError(
            "Install deps: pip install transformers torch sentencepiece openpyxl"
        ) from e

    # --- read data (simple & robust) ---
    def _read_any(p: Path) -> pd.DataFrame:
        if p.suffix.lower() in {".xlsx", ".xls"}:
            return pd.read_excel(p)
        if p.suffix.lower() == ".csv":
            for enc in ("utf-8-sig", "cp1256", "utf-16"):
                try:
                    df = pd.read_csv(p, encoding=enc, engine="python")
                    if df.shape[1] == 1 and enc == "utf-16":
                        df = pd.read_csv(p, encoding="utf-16", sep="\t", engine="python")
                    # If semicolon header got merged, re-read with ';'
                    if df.shape[1] == 1 and isinstance(df.columns[0], str) and ";" in df.columns[0]:
                        df = pd.read_csv(p, encoding=enc, sep=";", engine="python")
                    return df
                except Exception:
                    pass
        # Final fallback: maybe it's really an Excel file
        return pd.read_excel(p)

    df = _read_any(in_path)
    df.columns = df.columns.astype(str).str.strip()

    # --- columns ---
    if fa_col not in df.columns:
        raise KeyError(f"Column '{fa_col}' not found. Available: {list(df.columns)}")
    if en_col not in df.columns:
        df[en_col] = ""

    # Ensure helper columns: S, I, D, N, WER
    base5 = ["S", "I", "D", "N", "WER"]
    if extra_columns is None:
        cols_needed = base5
    else:
        extra = list(extra_columns[:5])
        cols_needed = extra + base5[len(extra):]
    for c in cols_needed:
        if c not in df.columns:
            df[c] = ""

    # --- line range (1-based inclusive) ---
    n = len(df)
    if n == 0:
        raise ValueError("Input is empty.")
    start_line = max(1, start_line)
    end_line = n if end_line is None else min(end_line, n)
    if start_line > end_line:
        raise ValueError(f"start_line ({start_line}) must be <= end_line ({end_line})")
    s, e = start_line - 1, end_line - 1

    # --- translator (NLLB) ---
    device = 0 if torch.cuda.is_available() else -1
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    translator = pipeline(
        "translation",
        model=model_name,
        tokenizer=tokenizer,
        device=device,
    )
    SRC, TGT = "pes_Arab", "eng_Latn"   # Persian → English (NLLB language codes)

    # --- translate in batches ---
    texts_fa = df.loc[s:e, fa_col].fillna("").astype(str).tolist()
    texts_en: List[str] = []
    for i in range(0, len(texts_fa), batch_size):
        batch = texts_fa[i:i+batch_size]
        outs = translator(batch, src_lang=SRC, tgt_lang=TGT, max_length=256)
        texts_en.extend([o["translation_text"] for o in outs])
    df.loc[s:e, en_col] = texts_en

    # --- save output ---
    if out_path is None:
        out_path = in_path.with_suffix(".fa_en.range.xlsx")
    elif out_path.exists() and out_path.is_dir():
        out_path = out_path / (in_path.stem + ".fa_en.range.xlsx")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(out_path, index=False)
    return str(out_path)


In [32]:
out = translate_fa_to_en_excel_range(
    in_path=r"D:\Block A 2\Whisper\metri_6.csv",
    fa_col="text_fa",
    en_col="text_en",
    start_line=105,
    end_line=165,
    extra_columns=["S","I","D","N","WER"],
)
print("Saved:", out)


Device set to use cpu
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
C:\Users\Asus\AppData\Local\Temp\ipykernel_26108\2162229878.py:102: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['If you answered correctly,', "And your wife's going to jail.", 'My kids are right.', "We've got these all over the place, which makes the Qur'an so hard.", "You thought I'd release a woman for the kids.", 'What does the judge read?', 'The report.', "Who's writing the report?", 'I did.', 'What did I write?', 'There they are.', "I've written about female sex owners doing the same thing.", 'It demands that the light be nothing.', 'She keeps a few years.', "What's going to happen to kids in a few years?", 'What am I supposed to do?', "If I'm goi

Saved: D:\Block A 2\Whisper\metri_6.fa_en.range.xlsx


In [35]:
out = translate_fa_to_en_excel_range(
    in_path=r"D:\Block A 2\Whisper\metri_6.csv",
    fa_col="text_fa",
    en_col="text_en",
    start_line=227,
    end_line=287,
    extra_columns=["S","I","D","N","WER"],
    out_path=r"D:\Block A 2\Whisper\_out\metri_6.fa_en.range.xlsx",  # new file
)





Device set to use cpu
C:\Users\Asus\AppData\Local\Temp\ipykernel_26108\2162229878.py:102: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '["You don't even know his blood.", "I don't even know a kid.", 'How could you have sex with him?', "You've always been there.", 'The passenger car is more loaded.', "You don't exactly say everything we've got belongs to him.", "What's the matter?", "If we don't get it, it's yours.", "When we have four, we say it's his.", 'Tell me what difference four hundred people make when you know who.', 'Every son of his own made time to greet him.', 'Nasser the Count knows how to escape.', "That's right.", 'No matter how you treat them, they should treat you the same way.', 'What do they call these three sidewalks?', "It's called Band.", 'What do they say to these three sidewalks?', "It's called the formation of the band.", "What's the formation order?", 'The execution.', "What's wi

In [36]:
out = translate_fa_to_en_excel_range(
    in_path=r"D:\Block A 2\Whisper\metri_6.csv",
    fa_col="text_fa",
    en_col="text_en",
    start_line=305,
    end_line=365,
    extra_columns=["S","I","D","N","WER"],
    out_path=r"D:\Block A 2\Whisper\_out\metri_6.fa_en.range3.xlsx",  # new file
)


Device set to use cpu
C:\Users\Asus\AppData\Local\Temp\ipykernel_26108\2162229878.py:102: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['At the door.', "You're going to repeat it.", 'Where is he now?', "I haven't seen him in a year and a half.", 'You were married to Nazar the Grey?', "No, it wasn't, but show me.", "What's the matter?", 'We had a disagreement. I split up.', 'Disagreements?', "No, it's not.", 'More class differences.', 'So you were more knowledgeable than he was?', 'No need for money.', 'And what is the moral requirement?', 'No, he was very polite.', "It's more of a problem.", 'It was his personality to his morality.', "I don't know what you mean.", 'Explain it right.', "I'm getting married soon.", "I still don't understand why I'm here.", "I don't know why I have to answer these questions.", 'From now on, every time you fall in love.', "I'll have you detained for a week.", 'And then they